# TN1 phần A — LSTM làm mốc

Chạy **song song** với `TN1_TCN_DSTCN.ipynb` ở một phiên Colab khác. Hai notebook độc lập hoàn toàn, không cần chờ nhau.

| notebook | chạy gì | thời gian |
|---|---|---|
| **TN1_LSTM.ipynb** ← đang mở | LSTM: 4 fold CV, rồi 3 seed test GHIJ | ~3 giờ |
| TN1_TCN_DSTCN.ipynb | TCN-64 và DS-TCN-64: 4 fold CV mỗi cái | ~2.2 giờ |
| TN1.ipynb | gộp kết quả, so sánh, chạy GHIJ cho kiến trúc thắng | ~1.5 giờ |

## Vì sao LSTM phải chạy CV

Số LSTM ở TN0 (`0.822642`) đo trên **G H I J**, train đủ **8 người**. Số TN1 đo trên **4 fold của A B C D E F K L**, mỗi model train **6 người**. Khác người test, khác lượng dữ liệu, khác giao thức — đặt cạnh nhau là vô nghĩa.

Muốn nói "TCN hơn LSTM" thì LSTM phải được đo bằng đúng cái thước đó.

## Giao thức

Bốn fold cố định trên tám người `A B C D E F K L`, dùng y nguyên cho mọi thí nghiệm:

```
val_AB   train C D E F K L    chấm A B
val_CE   train A B D F K L    chấm C E
val_DF   train A B C E K L    chấm D F
val_KL   train A B C D E F    chấm K L
```

Cấu hình giữ nguyên như MobiVital công bố: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr_threshold` 0.9, không RevIN. Một seed.

Điểm chấm trên **buổi ghi thô**, model tự chọn kênh, không nhìn nhịp thở thật.


## 1. Chuẩn bị Colab


Mount Drive để lấy lại cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Tải mã nguồn rồi vào thư mục đó. `setup_colab.py` clone MobiVital và ghim commit `4319731d` — `src/mobivital_reference.py` mượn sáu hàm từ repo họ.


In [ ]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB — chỉ train trên cửa sổ đã cắt, chấm trên `by_user/*.npz`.


In [ ]:
!python scripts/restore_processed_data_on_drive.py


## 2. LSTM — 4 fold CV

Ghi 5 dòng vào `runs/summary.csv`: bốn dòng fold và một dòng `TONG` mang `cv_score` cùng `cv_std`.

Khoảng **1.5 giờ**: 0.85 phút mỗi epoch × 20 epoch × 4 fold, cộng chấm điểm 1289 buổi ghi.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model lstm


## 3. LSTM — mốc GHIJ, 3 seed

Train đủ tám người `A B C D E F K L` rồi test 537 buổi ghi của `G H I J` — đúng pipeline bài báo dùng, không phải model fold chỉ train 6 người.

Ba seed để báo cáo `mean ± std`. Một lần chạy cho một con số không phân biệt được hơn thật với hơn may: trọng số khởi tạo và thứ tự xáo trộn đổi theo seed.

Bước này **không phụ thuộc kết quả CV** nên chạy luôn được. Kiến trúc thắng sẽ chạy GHIJ ở `TN1.ipynb` sau.

Khoảng **1.5 giờ**.


In [ ]:
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 0
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 1
!python scripts/run_final_test.py --experiment tn1_ghij --model lstm --seed 2


## 4. Cất kết quả

Nén ra tên riêng `tn1_lstm.zip` để **không đè** tệp của phiên chạy TCN. Cả hai phiên đều ghi `runs/tn1/`; tệp điểm không đè nhau vì tên cấu hình khác nhau, nhưng `summary.csv` thì mỗi phiên chỉ có dòng của riêng mình.


In [ ]:
!cd runs && zip -qr /content/drive/MyDrive/mobivital/tn1_lstm.zip tn1 tn1_ghij summary.csv
!ls -la /content/drive/MyDrive/mobivital/
